In [ ]:
import pandas as pd

,Market,Region,Country,State,City,Employee ID,Customer ID,Order ID,Order Date,Year (OrderDate),...,Sub-Category,Segment,Ship Date,Ship Mode,Payment Type,Discount,Profit,Quantity,Sales,Shipping Cost
0,EMEA,EMEA,Hungary,Budapest,Budapest,OMISSE00431,OMISS000299,HU-2021-626797,2021-01-01,2021,...,Storage,Consumer,2021-01-06,Express Shipping,Square,0.00,30.16,4,67.96,9.06
1,EMEA,Africa,Algeria,Constantine,Constantine,OMISSE00434,OMISS000561,DZ-2021-132562,2021-01-01,2021,...,Storage,Consumer,2021-01-07,Standard Shipping,Credit,0.00,106.83,2,422.98,35.65
2,APAC,Oceania,Australia,New South Wales,Wagga Wagga,OMISSE00059,OMISS000269,AU-2021-699976,2021-01-01,2021,...,Furnishings,Consumer,2021-01-09,Standard Shipping,Credit,0.12,38.87,5,111.95,4.90
3,APAC,Oceania,Australia,New South Wales,Wagga Wagga,OMISSE00059,OMISS000269,AU-2021-300725,2021-01-01,2021,...,Paper,Consumer,2021-01-09,Standard Shipping,Credit,0.11,15.68,2,55.98,2.52
4,APAC,Oceania,Australia,New South Wales,Wagga Wagga,OMISSE00059,OMISS000475,AU-2021-341993,2021-01-01,2021,...,Supplies,Consumer,2021-01-09,Standard Shipping,Credit,0.11,37.26,3,120.97,10.06


In [6]:
file_path = "/Users/lucasben/Documents/mba-business-analytics/Financial Performance Analysis/data/OMIS Store.xlsx"

orders = pd.read_excel(file_path, sheet_name=0)
returns = pd.read_excel(file_path, sheet_name=1)
customers = pd.read_excel(file_path, sheet_name=2)
employees = pd.read_excel(file_path, sheet_name=3)
products = pd.read_excel(file_path, sheet_name=4)

In [ ]:
list_data = [orders, returns, customers, employees, products]

for df in list_data:
    print((df.isnull().sum() / len(df) * 100).round(2))

# 1.5% of product colour data is missing. I'll impute it with the most common colour for that product

Market              0.0
Region              0.0
Country             0.0
State               0.0
City                0.0
Employee ID         0.0
Customer ID         0.0
Order ID            0.0
Order Date          0.0
Year (OrderDate)    0.0
Order Priority      0.0
Product ID          0.0
Product Name        0.0
Category            0.0
Sub-Category        0.0
Segment             0.0
Ship Date           0.0
Ship Mode           0.0
Payment Type        0.0
Discount            0.0
Profit              0.0
Quantity            0.0
Sales               0.0
Shipping Cost       0.0
dtype: float64
Returned    0.0
Order ID    0.0
Market      0.0
dtype: float64
Customer ID       0.0
Customer Name     0.0
Sex               0.0
Market            0.0
Region            0.0
Rewards Member    0.0
dtype: float64
Employee ID       0.0
Employee Name     0.0
Market            0.0
Region            0.0
Title             0.0
Hire Date         0.0
Birth Date        0.0
Email Address     0.0
Marital Status    0.0
S

In [18]:
missing_colour = products[products['Colour'].isnull()]

missing_products = missing_colour['Product Name'].tolist()

missing_colour['Product Name'].unique()

array(['3M Office Air Cleaner', '3M Polarizing Light Filter Sleeves',
       "3M Replacement Filter for Office Air Cleaner for 20' x 33' Room",
       'American Pencil', 'Barrel Sharpener',
       'Berol Giant Pencil Sharpener', 'Binding Machine Supplies',
       'Blackstonian Pencils',
       'Boston 1645 Deluxe Heavier-Duty Electric Pencil Sharpener',
       'Boston 16701 Slimline Battery Pencil Sharpener',
       'Boston 16765 Mini Stand Up Battery Pencil Sharpener',
       'Boston 16801 Nautilus Battery Pencil Sharpener',
       'Boston 1730 StandUp Electric Pencil Sharpener',
       'Boston 1799 Powerhouse Electric Pencil Sharpener',
       'Boston 1827 Commercial Additional Cutter, Drive Gear & Gear Rack for 1606',
       'Boston 1900 Electric Pencil Sharpener',
       'Boston 19500 Mighty Mite Electric Pencil Sharpener',
       'Boston Heavy-Duty Trimline Electric Pencil Sharpeners',
       'Boston Home & Office Model 2000 Electric Pencil Sharpeners',
       'Boston KS Multi-Siz

In [28]:
product_colour_mode = products.groupby('Product Name')['Colour'].apply(lambda x: x.mode().iloc[0] if not x.mode().empty else 'Unknown').reset_index()

# .groupby('Product Name)['Colour] groups the data by unique product names and selects the colours from each group

# .apply(lambda x: ) applies a function to each group where x represents the colour series for each product group

# .iloc[0] if not x.mode().empty else 'Unknown' gest the first mode value and checks if there are any mode values, if no mode exists, returns  'Unknown'

# .reset_index() converts the result from a series into a dataframe

product_colour_mode.columns = ['Product Name', 'Mode_Colour']

print(product_colour_mode)

                                           Product Name Mode_Colour
0     "While you Were Out" Message Book, One Form pe...       Multi
1              #10 Gummed Flap White Envelopes, 100/Box       White
2                         #10 Self-Seal White Envelopes       White
3            #10 White Business Envelopes,4 1/8 x 9 1/2       White
4               #10- 4 1/8" x 9 1/2" Recycled Envelopes       White
...                                                 ...         ...
3792  iKross Bluetooth Portable Keyboard + Cell Phon...       White
3793                         iOttie HLCRIO102 Car Mount       Black
3794                                iOttie XL Car Mount       Black
3795  invisibleSHIELD by ZAGG Smudge-Free Screen Pro...       Clear
3796                 netTALK DUO VoIP Telephone Service       Black

[3797 rows x 2 columns]


In [27]:
products_imputed = products.merge(product_colour_mode, on='Product Name', how='left') # merging the mode data with the original data

products_imputed['Colour'] = products_imputed['Colour'].fillna(products_imputed['Mode_Colour']) # imputing missing product colours

products_imputed = products_imputed.drop('Mode_Colour', axis=1) # removing the temporary mode column